<div style="padding: 20px; background: linear-gradient(90deg, #0f766e 0%, #2563eb 100%); border-radius: 10px; color: white;">
    <h1 style="color: white; border-bottom: none;">Module 4.3: Indexing Strategies - Flat, IVF, and HNSW</h1>
    <p style="font-size: 1.2em; opacity: 0.9;">How vector stores trade off recall, latency, and memory at search time.</p>
</div>

---

### Course alignment and free-first stack

- Covers: Vector-store working internals and indexing strategies, especially Flat search, IVF, and HNSW.
- Runtime stack: FAISS CPU plus local Hugging Face sentence-transformers embeddings. No paid API key is required.
- Current LangChain pattern: local embeddings through `langchain_huggingface`; raw FAISS is used here because this lesson is about index mechanics, not retriever wrappers.


## 1. Why indexing strategies matter

A vector store does two jobs:

1. Store embeddings and metadata.
2. Search for nearest vectors quickly enough for user-facing retrieval.

The simplest search scans every vector. That is accurate, but it becomes expensive as the corpus grows. Production systems often use approximate nearest neighbor indexes that trade a small amount of recall for much lower latency.

| Strategy | Idea | Best for | Main trade-off |
| :--- | :--- | :--- | :--- |
| Flat | Compare the query with every vector | Small corpora, exact baselines | Slow at large scale |
| IVF | Cluster vectors, search only the closest clusters | Large corpora with tunable speed | Needs training and `nprobe` tuning |
| HNSW | Build a navigable graph of nearby vectors | Fast high-recall search | More memory and build-time cost |


In [ ]:
import os
import time
import numpy as np
import faiss
from langchain_huggingface import HuggingFaceEmbeddings

os.environ["TOKENIZERS_PARALLELISM"] = "false"

embedding_model = os.getenv("EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2")
embeddings = HuggingFaceEmbeddings(model_name=embedding_model)

corpus = [
    "RAG grounds language model answers in retrieved documents.",
    "LangGraph represents agentic workflows as stateful graphs.",
    "Chroma stores embeddings and metadata for local vector search.",
    "FAISS provides high-performance vector indexing algorithms.",
    "IVF uses coarse clusters to reduce the search space.",
    "HNSW creates a graph for fast approximate nearest neighbor search.",
    "BM25 is a sparse keyword retriever based on term frequency.",
    "Cross-encoders rerank candidate documents with query-document attention.",
]

query = "Which indexing method uses clusters before vector search?"

doc_vectors = np.array(embeddings.embed_documents(corpus), dtype="float32")
query_vector = np.array([embeddings.embed_query(query)], dtype="float32")

# Normalize for cosine similarity via inner product.
faiss.normalize_L2(doc_vectors)
faiss.normalize_L2(query_vector)

dimension = doc_vectors.shape[1]
print(f"Corpus vectors: {doc_vectors.shape}")


## 2. Flat index: exact baseline

`IndexFlatIP` performs an exhaustive scan. It is the baseline you compare approximate indexes against because it gives exact nearest neighbors for the chosen metric.


In [ ]:
flat = faiss.IndexFlatIP(dimension)
flat.add(doc_vectors)

start = time.perf_counter()
flat_scores, flat_ids = flat.search(query_vector, k=3)
flat_ms = (time.perf_counter() - start) * 1000

print(f"Flat search latency: {flat_ms:.3f} ms")
for rank, (idx, score) in enumerate(zip(flat_ids[0], flat_scores[0]), start=1):
    print(f"{rank}. score={score:.3f} | {corpus[idx]}")


## 3. IVF index: coarse clusters plus local scan

IVF first trains coarse clusters. At query time, FAISS searches only the closest cluster lists. The `nprobe` setting controls how many lists are searched: higher `nprobe` improves recall but costs more latency.

Small teaching corpora are too tiny for a realistic IVF benchmark, but the mechanics are the same at production scale.


In [ ]:
nlist = 2  # number of coarse clusters; production values are much larger
quantizer = faiss.IndexFlatIP(dimension)
ivf = faiss.IndexIVFFlat(quantizer, dimension, nlist, faiss.METRIC_INNER_PRODUCT)

ivf.train(doc_vectors)
ivf.add(doc_vectors)
ivf.nprobe = 1

start = time.perf_counter()
ivf_scores, ivf_ids = ivf.search(query_vector, k=3)
ivf_ms = (time.perf_counter() - start) * 1000

print(f"IVF search latency: {ivf_ms:.3f} ms with nprobe={ivf.nprobe}")
for rank, (idx, score) in enumerate(zip(ivf_ids[0], ivf_scores[0]), start=1):
    if idx == -1:
        continue
    print(f"{rank}. score={score:.3f} | {corpus[idx]}")


## 4. HNSW index: graph navigation

HNSW builds a graph where nearby vectors are connected. Search starts from entry points and walks the graph toward better neighbors. The `efSearch` parameter tunes recall vs latency, while `M` controls graph connectivity and memory use.


In [ ]:
m = 16
hnsw = faiss.IndexHNSWFlat(dimension, m, faiss.METRIC_INNER_PRODUCT)
hnsw.hnsw.efSearch = 32
hnsw.add(doc_vectors)

start = time.perf_counter()
hnsw_scores, hnsw_ids = hnsw.search(query_vector, k=3)
hnsw_ms = (time.perf_counter() - start) * 1000

print(f"HNSW search latency: {hnsw_ms:.3f} ms with efSearch={hnsw.hnsw.efSearch}")
for rank, (idx, score) in enumerate(zip(hnsw_ids[0], hnsw_scores[0]), start=1):
    print(f"{rank}. score={score:.3f} | {corpus[idx]}")


## 5. Practical selection guide

Use Flat when the corpus is small, when correctness matters more than latency, or when you need a reference result set.

Use IVF when you have enough vectors to train meaningful clusters and want a simple latency/recall knob through `nprobe`.

Use HNSW when you need strong recall with fast interactive latency and can afford extra memory for the graph.

In production RAG, always measure retrieval quality after changing index settings. Faster retrieval is not useful if the generator receives the wrong evidence.
